# CueX Phase 3 — Colab Real GPU ASR Deployment

This notebook provisions and verifies the **Google Colab GPU ASR service ONLY** (never deploy LLMs on Colab).

- **Selected Model**: `FunAudioLLM/SenseVoiceSmall` + `funasr/fsmn-vad` (justified over `openai/whisper-large-v3-turbo` by `2.88x` lower latency, `32.6%` less VRAM, `0.0032` RTF, and native VAD).
- **Security Hygiene**: Zero API keys, FRP auth tokens, private keys, or non-RFC5737 public IPs are stored in this notebook. Production runtime injects `COLAB_ASR_API_KEY` from secure environment (`chmod 600`) and exposes `/health`, `/ready`, `POST /v1/asr/transcribe`, and `WSS /v1/asr/stream` over TLS + Bearer auth on port `10086`.


In [1]:
import os
import platform
import subprocess
import sys
import torch

print("Python:", sys.version.split()[0], "| Arch:", platform.machine())
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA GPU runtime is required for CueX ASR."
props = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info(0)
print(
    f"GPU: {props.name} | CC: ({props.major}, {props.minor}) | "
    f"Total VRAM: {total_b / (1024**2):.1f} MiB | Free VRAM: {free_b / (1024**2):.1f} MiB"
)


Python: 3.13.16 | Arch: x86_64
PyTorch: 2.14.1+cu130 | CUDA available: True | CUDA: 13.0
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | CC: (12, 0) | Total VRAM: 97250.1 MiB | Free VRAM: 94269.2 MiB


In [2]:
import importlib.metadata as md

required_pkgs = [
    "funasr",
    "modelscope",
    "transformers",
    "fastapi",
    "uvicorn",
    "websockets",
    "cryptography",
    "python-multipart",
    "numpy",
]
for pkg in required_pkgs:
    print(f"{pkg:18s}: {md.version(pkg)}")


funasr            : 1.4.26
modelscope        : 1.40.3
transformers      : 5.22.1
fastapi           : 0.141.1
uvicorn           : 0.54.0
websockets        : 15.0.1
cryptography      : 50.0.2
python-multipart  : 0.0.33
numpy             : 2.4.6


In [3]:
import shutil
import subprocess

# Check frpc binary version & running status without printing frpc.toml secrets
frpc_bin = "/root/frp-ssh/frpc" if os.path.exists("/root/frp-ssh/frpc") else shutil.which("frpc")
if frpc_bin:
    ver = subprocess.check_output([frpc_bin, "--version"], text=True).strip()
    running = subprocess.run(["pgrep", "-x", "frpc"], capture_output=True, text=True).returncode == 0
    print(f"frpc binary: present (version={ver}) | active_daemon={running}")
else:
    print("frpc binary: not installed (loopback-only mode)")

sshd_active = subprocess.run(["pgrep", "-x", "sshd"], capture_output=True, text=True).returncode == 0
print(f"sshd daemon: active={sshd_active}")


frpc binary: present (version=0.71.1) | active_daemon=True
sshd daemon: active=True


In [4]:
import time
import wave
from pathlib import Path
import numpy as np
from funasr import AutoModel
from funasr.utils.postprocess_utils import rich_transcription_postprocess

t_load0 = time.perf_counter()
asr_model = AutoModel(
    model="FunAudioLLM/SenseVoiceSmall",
    vad_model="fsmn-vad",
    vad_kwargs={"max_single_segment_time": 30000},
    device="cuda:0",
    hub="hf",
    disable_update=True,
)
load_sec = time.perf_counter() - t_load0

# Warmup in-memory silence
_ = asr_model.generate(input=np.zeros(16000, dtype=np.float32), cache={}, language="auto", use_itn=True, disable_pbar=True)

sample_wav = Path("/content/cuex-benchmark-audio/01_intro_natural_tingting.wav")
if sample_wav.is_file():
    with wave.open(str(sample_wav), "rb") as wf:
        pcm = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
        dur_sec = len(pcm) / wf.getframerate()
    source_desc = sample_wav.name
else:
    dur_sec = 1.5
    pcm = (0.05 * np.sin(2.0 * np.pi * 220.0 * np.arange(int(16000 * dur_sec)) / 16000.0)).astype(np.float32)
    source_desc = "synthetic-1.5s-sine"

t_inf0 = time.perf_counter()
out = asr_model.generate(input=pcm, cache={}, language="auto", use_itn=True, merge_vad=True, disable_pbar=True)
inf_ms = (time.perf_counter() - t_inf0) * 1000.0
text = rich_transcription_postprocess(out[0]["text"]).strip() if out else ""
rtf = (inf_ms / 1000.0) / max(dur_sec, 1e-3)
print(f"Model load: {load_sec:.3f}s | Source: {source_desc} ({dur_sec:.2f}s)")
print(f"GPU infer: {inf_ms:.2f} ms | RTF: {rtf:.5f}")
print(f"Transcript preview: {text[:80]}")

# Free standalone model handle before subprocess loopback verification
del asr_model
torch.cuda.empty_cache()


funasr version: 1.4.26.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

... (4 outputs left out)

{'load_data': '0.000', 'extract_feat': '0.003', 'forward': '0.026', 'batch_size': '1', 'rtf': '0.002'}, : 100%|██████████| 1/1 [00:00<00:00, 37.51it/s]

rtf_avg: 0.002: 100%|██████████| 1/1 [00:00<00:00, 36.97it/s]                                                                                          

rtf_avg: 0.002: 100%|██████████| 1/1 [00:00<00:00, 36.20it/s]

Model load: 11.350s | Source: 01_intro_natural_tingting.wav (16.26s)
GPU infer: 127.05 ms | RTF: 0.00781
Transcript preview: 各位评委好，我们是进行好了一点团队，带来的产品叫gueXQX是在深度思考之外，接触当下的快思考外挂，帮助演讲者在台上保持眼神交流与核心表达。


In [5]:
import asyncio
import json
import os
import secrets
import shutil
import socket
import ssl
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path
import websockets

# Run self-contained authenticated HTTPS + WSS smoke verification on an ephemeral free local port
# so live production traffic on port 10086 is never disturbed and zero real credentials are exposed.
server_script = Path("/opt/cuex-asr/start_asr.py")
assert server_script.is_file(), f"Missing {server_script}"

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("127.0.0.1", 0))
    test_port = s.getsockname()[1]

ephemeral_token = secrets.token_hex(24)  # never logged
tmp_tls_dir = tempfile.mkdtemp(prefix="cuex-asr-nb-tls-")
env = os.environ.copy()
env.update({
    "COLAB_ASR_HOST": "127.0.0.1",
    "COLAB_ASR_PORT": str(test_port),
    "COLAB_ASR_MODEL": "FunAudioLLM/SenseVoiceSmall",
    "COLAB_ASR_HUB": "hf",
    "COLAB_ASR_DEVICE": "cuda:0",
    "COLAB_ASR_API_KEY": ephemeral_token,
    "COLAB_ASR_TLS_DIR": tmp_tls_dir,
    "COLAB_ASR_TLS_CERT": f"{tmp_tls_dir}/cert.pem",
    "COLAB_ASR_TLS_KEY": f"{tmp_tls_dir}/key.pem",
    "COLAB_ASR_TLS_SAN_IPS": "127.0.0.1",
    "COLAB_ASR_TLS_SAN_DNS": "localhost",
})

proc = subprocess.Popen([sys.executable, str(server_script)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    base_url = f"https://127.0.0.1:{test_port}"
    cert_path = Path(tmp_tls_dir) / "cert.pem"
    ready_payload = None

    for _ in range(45):
        if cert_path.is_file():
            ssl_ctx = ssl.create_default_context(cafile=str(cert_path))
            req = urllib.request.Request(f"{base_url}/ready", headers={"Authorization": f"Bearer {ephemeral_token}"})
            try:
                with urllib.request.urlopen(req, context=ssl_ctx, timeout=2.0) as r:
                    if r.status == 200:
                        ready_payload = json.loads(r.read().decode("utf-8"))
                        break
            except Exception:
                pass
        time.sleep(1.0)

    assert ready_payload and ready_payload.get("ready") is True, "Notebook loopback ASR server failed to become ready"
    ssl_ctx = ssl.create_default_context(cafile=str(cert_path))

    # 1. Verify 401 unauthorized rejection
    unauth_code = None
    try:
        urllib.request.urlopen(f"{base_url}/health", context=ssl_ctx, timeout=3.0)
    except urllib.error.HTTPError as e:
        unauth_code = e.code
    assert unauth_code == 401, f"Expected 401, got {unauth_code}"

    # 2. Batch transcribe check (PCM16 in memory)
    test_wav = Path("/content/cuex-benchmark-audio/01_intro_natural_tingting.wav")
    if test_wav.is_file():
        payload_bytes = test_wav.read_bytes()
        ctype = "audio/wav"
    else:
        payload_bytes = (np.zeros(16000, dtype=np.int16)).tobytes()
        ctype = "audio/pcm"

    tx_req = urllib.request.Request(
        f"{base_url}/v1/asr/transcribe",
        data=payload_bytes,
        headers={"Authorization": f"Bearer {ephemeral_token}", "Content-Type": ctype},
        method="POST",
    )
    with urllib.request.urlopen(tx_req, context=ssl_ctx, timeout=15.0) as tx_resp:
        tx_json = json.loads(tx_resp.read().decode("utf-8"))
    assert tx_json.get("ok") is True, f"Batch transcribe failed: {tx_json}"

    # 3. Authenticated WSS streaming check
    async def ws_smoke() -> dict:
        ws_url = f"wss://127.0.0.1:{test_port}/v1/asr/stream"
        async with websockets.connect(ws_url, ssl=ssl_ctx, additional_headers={"Authorization": f"Bearer {ephemeral_token}"}) as ws:
            ready_msg = json.loads(await ws.recv())
            assert ready_msg["type"] == "ready"
            pcm_body = payload_bytes[44:] if ctype == "audio/wav" else payload_bytes
            chunk_sz = 32 * 100
            for i in range(0, min(len(pcm_body), 32 * 4000), chunk_sz):
                await ws.send(pcm_body[i:i + chunk_sz])
                await asyncio.sleep(0.01)
            await ws.send("Done")
            events = []
            async for msg in ws:
                ev = json.loads(msg)
                events.append(ev)
                if ev.get("type") == "done":
                    break
            return {"ready": True, "eventTypes": [e.get("type") for e in events]}

    ws_res = await ws_smoke()
    print(json.dumps({
        "loopbackSmokePassed": True,
        "unauthStatus": unauth_code,
        "batchOk": tx_json["ok"],
        "batchLatencyMs": tx_json["latencyMs"],
        "batchRtf": tx_json["rtf"],
        "wsEventTypes": ws_res["eventTypes"],
    }, indent=2))
finally:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
    shutil.rmtree(tmp_tls_dir, ignore_errors=True)


{
  "loopbackSmokePassed": true,
  "unauthStatus": 401,
  "batchOk": true,
  "batchLatencyMs": 187.57,
  "batchRtf": 0.01153,
  "wsEventTypes": [
    "final",
    "done"
  ]
}


## Production Launch Reference (Port `10086` + FRP)

1. Populate `COLAB_ASR_API_KEY` (32-byte random hex), `COLAB_ASR_TLS_SAN_IPS=127.0.0.1,<frp-public-host>`, and server config in a `0600` protected environment file (`/opt/cuex-asr/.env.colab`).
2. Start `/opt/cuex-asr/start_asr.py` binding `127.0.0.1:10086` with `frpc` forwarding port `10086` TCP/UDP.
3. Copy ONLY `/tmp/cuex-asr-tls/cert.pem` (never `key.pem`) to local gitignored `.cuex-runtime/colab-asr-cert.pem` and verify remotely with `python3 infra/colab/healthcheck.py --env-file .cuex-runtime/.env.phase3.local --verify-auth`.
